### Create dimension tables first:
* team
* drivers
* circuit
* race



In [0]:
tables=spark.catalog.listTables("f1_warehouse.silver")

temp_gold={}

for table in tables:
    df=spark.table(f"f1_warehouse.silver.{table.name}")
    temp_gold[table.name]=df

### Create mappings

In [0]:
for table_names,df in temp_gold.items():
    print(table_names)

In [0]:
from pyspark.sql.functions import (col,upper,trim,concat_ws,coalesce,lit,udf,when,max,create_map,to_timestamp,date_sub,to_date,substring)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from datetime import date


### Create dimension table for team

In [0]:
temp_gold["kaggle_constructors"].show(5)

In [0]:
temp_gold["openf1_drivers"].show(5)

In [0]:
# we will join on name/team_name 

k_teams=temp_gold["kaggle_constructors"].withColumn("team_name",upper(trim(col("name"))))

of1_teams=temp_gold["openf1_drivers"].withColumn("team_name",upper(trim(col("team_name"))))

# ensure no duplicates first for of1

of1_teams=of1_teams.select("team_name","team_colour").dropDuplicates(["team_name"])

# change red bull racing to red bull and alpint to alpine f1 

of1_teams=of1_teams.withColumn(
    "team_name",
    when(col("team_name") == "RED BULL RACING", "RED BULL")
    .when(col("team_name") == "ALPINE", "ALPINE F1 TEAM")
    .otherwise(col("team_name"))
)

# join both tables

teams_dim=(
    k_teams.alias("k").join(
        of1_teams.alias("o"),
        on="team_name",
        how="full_outer"))




In [0]:
# rows in openf1(left table) which dont have a match in kaggle n (right table)


of1_teams.join(
    k_teams,
    on="team_name",
    how="left_anti" 
).select("team_name").show(100, truncate=False)

In [0]:

# kick sauber,alpine,rb and red bull racing exist before 2024 so check kaggle set 

teams_dim.filter(
    col("team_name").contains("RED")|
    col("team_name").contains("ALPINE")|
    col("team_name").contains("SAUBER")|
    col("team_name").contains("TORO")|
    col("team_name").contains("ALPHA")
).show(truncate=False)

In [0]:

# drop Alpine since there's alrdy Alpine F1 Team
# drop red bull racing 
# drop null
# kick sauber is different from bmw sauber so keep it


teams_dim=teams_dim.filter(col("team_name").isNotNull())



In [0]:
#select useful column names

teams_dim=teams_dim.select("team_name","constructor_id","name","nationality","team_colour","url")

In [0]:
#check for duplicates in team_name

teams_dim.groupBy("team_name").count().filter(col("count")>1).show()

In [0]:
teams_dim.filter(col("constructor_id").isNull()).show()

In [0]:
#create constructor id for remaining teams
#unionbyname needs to have the same columns

window=Window.orderBy("team_name")

max_constructor_id=teams_dim.select(max("constructor_id")).first()[0]

new_teams=teams_dim.filter(col("constructor_id").isNull()).withColumn("constructor_id",row_number().over(window)+max_constructor_id)

existing_teams=teams_dim.filter(col("constructor_id").isNotNull())

teams_dim=existing_teams.unionByName(new_teams)

In [0]:
#check for duplicates or null values
print(teams_dim.filter(col("constructor_id").isNull()).count())

teams_dim.groupBy("constructor_id").count().filter(col("count") > 1).show()

teams_dim.groupBy("team_name").count().filter(col("count") > 1).show()

In [0]:
teams_dim=teams_dim.withColumnRenamed("constructor_names","team_names")

In [0]:

teams_dim.filter(col("name").isNull()).show()

In [0]:
teams_dim = (
    teams_dim
    .withColumn(
        "name",
        when(col("team_name") == "AUDI", "Audi")
        .when(col("team_name") == "CADILLAC", "Cadillac")
        .when(col("team_name") == "KICK SAUBER", "Kick Sauber")
        .when(col("team_name") == "RACING BULLS", "Racing Bulls")
        .when(col("team_name") == "RB", "RB")
        .otherwise(col("name"))
    )
    .withColumn(
        "nationality",
        when(col("team_name") == "AUDI", "German")
        .when(col("team_name") == "CADILLAC", "American")
        .when(col("team_name") == "KICK SAUBER", "Swiss")
        .when(col("team_name") == "RACING BULLS", "Italian")
        .when(col("team_name") == "RB", "Italian")
        .otherwise(col("nationality"))
    )
)

In [0]:
teams_dim.filter(col("name").contains("Alpine")).show()

In [0]:
#change name to red bbull racing
#change alpine f1 team to alpine

teams_dim = (
    teams_dim
    .withColumn(
        "name",
        when(col("name") == "Red Bull", "Red Bull Racing")
        .when(col("name") == "Alpine F1 Team", "Alpine")
        .otherwise(col("name"))
    )
)
                   

In [0]:
#save to gold
teams_dim.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("f1_warehouse.gold.dim_teams")

### Create dimension table for drivers version 2

* append list of newer 2025/2026 drivers to kaggle driver dataset

In [0]:
k_drivers=temp_gold["kaggle_drivers"]

In [0]:
k_drivers.show(5)

In [0]:
#create list of new 2025/2026 drivers

k_drivers.filter(col("surname").isin("Bearman","Antonelli","Antonelli","Hadjar","Doohan","Lindblad","Bortoleto","Lawson","Norris")).show()




In [0]:
from pyspark.sql.functions import udf
import unicodedata

remove_accents = udf(
    lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    if x else x)


k_drivers = k_drivers.withColumn("surname",remove_accents(col("surname")))

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import max, col
from datetime import date

# Keep only the 9 columns we need
k_drivers = k_drivers.select(
    "driver_id",
    "driver_ref",
    "number",
    "code",
    "forename",
    "surname",
    "dob",
    "nationality",
    "url"
)

# Get current maximum driver_id
max_driver_id = k_drivers.select(
    max("driver_id")
).collect()[0][0]

# Create new driver records
new_driver_rows = [
    Row(
        driver_id=max_driver_id + 1,
        driver_ref="kimi",
        number=12,
        code="ANT",
        forename="Kimi",
        surname="Antonelli",
        dob=date(2006, 8, 25),
        nationality="Italian",
        url="https://en.wikipedia.org/wiki/Kimi_Antonelli"
    ),
    Row(
        driver_id=max_driver_id + 2,
        driver_ref="isack",
        number=6,
        code="HAD",
        forename="Isack",
        surname="Hadjar",
        dob=date(2004, 9, 28),
        nationality="French",
        url="https://en.wikipedia.org/wiki/Isack_Hadjar"
    ),
    Row(
        driver_id=max_driver_id + 3,
        driver_ref="lindblad",
        number=41,
        code="LIN",
        forename="Arvid",
        surname="Lindblad",
        dob=date(2007, 9, 8),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Arvid_Lindblad"
    ),
    Row(
        driver_id=max_driver_id + 4,
        driver_ref="bortoleto",
        number=5,
        code="BOR",
        forename="Gabriel",
        surname="Bortoleto",
        dob=date(2004, 10, 4),
        nationality="Brazilian",
        url="https://en.wikipedia.org/wiki/Gabriel_Bortoleto"
    )

]

# Now k_drivers has 9 columns,
# so its schema matches the 9 fields in each Row
new_drivers_df = spark.createDataFrame(
    new_driver_rows,
    schema=k_drivers.schema
)

# Union the two DataFrames
k_drivers = k_drivers.unionByName(new_drivers_df)

# Test
k_drivers.filter(
    col("surname").contains("Hulkenberg")
).show()

In [0]:
k_drivers.filter(col("code").contains("HUL")).show()


In [0]:
# add binary column for existing drivers

current_driver_codes = [
    "NOR", "PIA",
    "RUS", "ANT",
    "VER", "HAD",
    "LEC", "HAM",
    "ALO", "STR",
    "GAS", "COL",
    "OCO", "BEA",
    "LAW", "LIN",
    "ALB", "SAI",
    "HUL", "BOR",
    "PER", "BOT"
]

k_drivers=k_drivers.withColumn("is_current_driver",when(col("code").isin(current_driver_codes),lit(1)).otherwise(lit(0)))


In [0]:
k_drivers.groupBy("code").count().filter(col("count")>1).show()


In [0]:
#check for remaining code duplicates
k_drivers.filter(col("code").isin("MSC", "BIA", "HAR", "VER", "DOO", "ALB", "MAG","da ")).show()

In [0]:
# change is current driver for duplicates
# chang  is current driver for lando norris number 4

k_drivers=k_drivers.withColumn("is_current_driver",when(col("driver_id").isin(27,818,830,846,860),0).otherwise(col("is_current_driver")))


In [0]:

#scd type 2 for verstappen,jack doohan,oliver bearman,isack hadjar,liam lawson,franco colapinto, lando norris whose driver numbers changed 

max_driver_id = k_drivers.select(
 max("driver_id")
).collect()[0][0]

new_numbers=[
    Row(
        driver_id=max_driver_id+1,
        driver_ref="verstappen",
        number=1,
        code="VER",
        forename="Max",
        surname="Verstappen",
        dob=date(1997, 9, 30),
        nationality="Dutch",
        url="https://en.wikipedia.org/wiki/Max_Verstappen",
        is_current_driver=1
    ),
    Row(
        driver_id=max_driver_id+2,
        driver_ref="verstappen",
        number=3,
        code="VER",
        forename="Max",
        surname="Verstappen",
        dob=date(1997, 9, 30),
        nationality="Dutch",
        url="https://en.wikipedia.org/wiki/Max_Verstappen",
        is_current_driver=0
        
    ),
    Row(
        driver_id=max_driver_id+3,
        driver_ref="norris",
        number=1,
        code="NOR",
        forename="Lando",
        surname="Norris",
        dob=date(1999,11, 13),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Lando_Norris",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id+4,
        driver_ref="bearman",
        number=87,
        code="BEA",
        forename="Oliver",
        surname="Bearman",
        dob=date(2005, 5, 8),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Oliver_Bearman",
        is_current_driver=1
    ),
    Row(
        driver_id=max_driver_id+5,
        driver_ref="doohan",
        number=7,
        code="DOO",
        forename="Jack",
        surname="Doohan",
        dob=date(2003,1, 20),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Jack_Doohan",
        is_current_driver=0
    ),
    Row(
        driver_id=max_driver_id + 6,
        driver_ref="bearman",
        number=50,
        code="BEA",
        forename="Oliver",
        surname="Bearman",
        dob=date(2005, 5, 8),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Oliver_Bearman",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id + 7,
        driver_ref="isack",
        number=36,
        code="HAD",
        forename="Isack",
        surname="Hadjar",
        dob=date(2004, 9, 28),
        nationality="French",
        url="https://en.wikipedia.org/wiki/Isack_Hadjar",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id + 8,
        driver_ref="isack",
        number=37,
        code="HAD",
        forename="Isack",
        surname="Hadjar",
        dob=date(2004, 9, 28),
        nationality="French",
        url="https://en.wikipedia.org/wiki/Isack_Hadjar",
        is_current_driver=1
    ),
    
    Row(
        driver_id=max_driver_id + 9,
        driver_ref="isack",
        number=41,
        code="HAD",
        forename="Isack",
        surname="Hadjar",
        dob=date(2004, 9, 28),
        nationality="French",
        url="https://en.wikipedia.org/wiki/Isack_Hadjar",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id + 10,
        driver_ref="lindblad",
        number=37,
        code="LIN",
        forename="Arvid",
        surname="Lindblad",
        dob=date(2007, 9, 8),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Arvid_Lindblad",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id + 11,
        driver_ref="lindblad",
        number=36,
        code="LIN",
        forename="Arvid",
        surname="Lindblad",
        dob=date(2007, 9, 8),
        nationality="British",
        url="https://en.wikipedia.org/wiki/Arvid_Lindblad",
        is_current_driver=1
    ),


    Row(
        driver_id=max_driver_id + 12,
        driver_ref="lawson",
        number=40,
        code="LAW",
        forename="Liam",
        surname="Lawson",
        dob=date(2002, 2, 11),
        nationality="New Zealander",
        url="https://en.wikipedia.org/wiki/Liam_Lawson",
        is_current_driver=1
    ),

    Row(
        driver_id=max_driver_id + 13,
        driver_ref="colapinto",
        number=45,
        code="COL",
        forename="Franco",
        surname="Colapinto",
        dob=date(2003, 5, 27),
        nationality="Argentinian",
        url="https://en.wikipedia.org/wiki/Franco_Colapinto",
        is_current_driver=1
    )


    
]

new_drivers_df = spark.createDataFrame(
    new_numbers,
    schema=k_drivers.schema
)

k_drivers=k_drivers.unionByName(new_drivers_df)


In [0]:
k_drivers.filter(col("driver_id").isin(866)).show()

In [0]:
#save to gold

k_drivers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("f1_warehouse.gold.dim_driver")

### Create dimension table for circuits

In [0]:
temp_gold["kaggle_circuits"].show(5)

In [0]:
temp_gold["openf1_meetings"].show(5)

In [0]:
k_circuits=temp_gold["kaggle_circuits"]
of1_circuits=temp_gold["openf1_meetings"]


#keep only relevant cols for both
k_circuits=k_circuits.select("circuit_id","circuit_ref","name","location","country","lat","lng","alt")
of1_circuits=of1_circuits.select("circuit_short_name","country_code","country_name","location","circuit_type")

# remove duplicates for of1 
of1_circuits=of1_circuits.dropDuplicates(["circuit_short_name","location"])



#join by location
circuits_dim=k_circuits.alias("k").join(
    of1_circuits.alias("o"),
    on="location",
    how="full_outer")



In [0]:
of1_circuits.select("location").distinct().orderBy("location").show(100,truncate=False)

In [0]:
k_circuits.select("location").distinct().orderBy("location").show(100,truncate=False)

In [0]:
circuits_dim.show(5)

In [0]:
#cols in of1 circuits that dont match k_circutis 

of1_circuits.join(
    k_circuits,
    on="location",
    how="left_anti" 
).select("location").show(100, truncate=False)

In [0]:
#do mapping 

k_circuits.filter(
    col("location").contains("Spa")|
    col("location").contains("Lusail")|
    col("location").contains("Bahrain")|
    col("location").contains("Yas")|
    col("location").contains("Monte")|
    col("location").contains("Monaco")|
    col("location").contains("Miami")
).show(truncate=False)

In [0]:
#map Monte Carlo or Monaco to Monte-Carlo
#map Spa Francorchamps to Spa
#map Miami Gardens to Miami
#map Sakhir to Bahrain
#map lusail to Al Daayen
#map yas marina,yas circuit to abu dhabi



of1_circuits=of1_circuits.withColumn("location",
     when(col("location") == "Spa-Francorchamps", "Spa")
    .when(col("location") == "Lusail", "Al Daayen")
    .when(col("location") == "Bahrain", "Sakhir")
    .when(col("location").isin("Yas Island","Yas Marina"),"Abu Dhabi")
    .when(col("location") == "Miami Gardens", "Miami")
    .when(col("location").isin("Monte Carlo","Monaco"), "Monte-Carlo")
    .when(col("location") == "Montréal", "Montreal")
    .otherwise(col("location")) )

In [0]:
#join agian by location


circuits_dim=k_circuits.alias("k").join(
    of1_circuits.alias("o"),
    on="location",
    how="full_outer")

In [0]:
circuits_dim.show(5)

In [0]:
#check for null values
circuits_dim.filter(
    col("circuit_id").isNull()|
    col("location").isNull()
).show()


In [0]:
#check for duplicates
circuits_dim.groupBy("location").count().filter(col("count") > 1).show()

In [0]:
#drop duplicates 

circuits_dim=circuits_dim.dropDuplicates(["location"])

In [0]:
#check for duplicates
circuits_dim.groupBy("circuit_id").count().filter(col("count") > 1).show()

In [0]:
circuits_dim = circuits_dim.select(
    "location",
    "circuit_id",
    "circuit_ref",
    "name",
    "country",
    "circuit_type",
    "lat",
    "lng",
    "alt"
)

In [0]:
circuits_dim.filter(col("location").isin("Madrid")).show()

In [0]:
#adding new locations aka madring in 2026


max_location_id=circuits_dim.select(max("circuit_id")).first()[0]


new_circuit=spark.createDataFrame([
    Row(
        location="Madrid",
        circuit_id=max_location_id+1,
        circuit_ref="madring",
        name="Madring",
        country="Spain",
        circuit_type="Temporary - Street",
        lat=40.2755,
        lng=-3.6152,
        alt=625

    )
])

circuits_dim=circuits_dim.unionByName(new_circuit)

In [0]:
#map circuit type 

In [0]:
# save to gold

circuits_dim.write.format("delta").mode("overwrite").saveAsTable("f1_warehouse.gold.dim_circuits")

### Create dimension table for meetings

In [0]:
temp_gold["kaggle_races"].show(5)

In [0]:
temp_gold["openf1_meetings"].show(5)

In [0]:

# cols to have to have for meetings_dim: year,round, meeting_name/name,date_start,date_end,is_cancelled,country_name,circuit_id(from circuit_dims),meeting_key(surrogate_key)

k_meetings=temp_gold["kaggle_races"]
of1_meetings=temp_gold["openf1_meetings"]

#create round column for openf1

window=Window.partitionBy("year").orderBy("date_start")


of1_meetings=of1_meetings.withColumn("round",when(col("meeting_name").isin("Pre Season Testing"),0).otherwise(row_number().over(window)-1))





In [0]:
#check if there are null dates which means that the race was cancelled for kaggle

k_meetings.filter(col("date").isNull()).count()

#since no cancelled races map all to false for new col "is_cancelled"

k_meetings=k_meetings.withColumn("is_cancelled",lit(False))

In [0]:
# create date start and date end for kaggle races 
#standardise date end and date start for openf1 races 
#concat_ws means concat with seperator

k_meetings=(k_meetings.withColumn("date_end",col("date"))
            .withColumn("date_start",date_sub(col("date"),2)))

of1_meetings=(of1_meetings.withColumn("date_start",to_date(col("date_start")))
             .withColumn("date_end",to_date(col("date_end"))))
                                                      


In [0]:
k_meetings.show(5)

In [0]:
of1_meetings.show(5)

In [0]:
#join circuit id for of1_meetings
of1_meetings=of1_meetings.withColumn("location",
     when(col("location") == "Spa-Francorchamps", "Spa")
    .when(col("location") == "Lusail", "Al Daayen")
    .when(col("location") == "Bahrain", "Sakhir")
    .when(col("location").isin("Yas Island","Yas Marina"),"Abu Dhabi")
    .when(col("location") == "Miami Gardens", "Miami")
    .when(col("location").isin("Monte Carlo","Monaco"), "Monte-Carlo")
    .when(col("location") == "Montréal", "Montreal")
    .otherwise(col("location")) )

of1_meetings = (
    of1_meetings.alias("o")
    .join(
        circuits_dim.select("location", "circuit_id").alias("c"),
        on="location",
        how="left"
    )
    .select(
        "o.*", #keeps all of1_meeting cols
        col("c.circuit_id").alias("new_circuit_id")
    )
    .drop("circuit_id")
    .withColumnRenamed("new_circuit_id", "circuit_id")
)


In [0]:
# join both tables 

k_meetings=k_meetings.select("circuit_id","race_id","year","round","name","is_cancelled","date_start","date_end")
of1_meetings=of1_meetings.select("circuit_id","meeting_id","year","round","meeting_name","is_cancelled","date_start","date_end")



In [0]:
# use circuit id and since theres only 1 f1 at a given circuit in a year
k_meetings=(k_meetings.withColumn("meeting_id",lit(None).cast("int"))
            .withColumnRenamed("name","meeting_name")) # changes null literal type to null integer type

of1_meetings=(of1_meetings.withColumn("race_id",lit(None).cast("int")))

#concat both datasets

cols=["meeting_id","race_id","circuit_id","year","round","meeting_name","is_cancelled","date_start","date_end"]

meetings_dim=(k_meetings.select(cols).unionByName(of1_meetings.select(cols)))

In [0]:
#data quality checks

print(meetings_dim.filter(col("meeting_name").isNull()).count())

print(meetings_dim.filter(col("circuit_id").isNull()).count())

#check duplicates
meetings_dim.groupBy("circuit_id","year","round").count().filter(col("count")>1).show()


In [0]:
#create surrogate meeting key

window=Window.orderBy("year","round","circuit_id") #1 circuit each year

meetings_dim=meetings_dim.withColumn("meeting_key",row_number().over(window))

In [0]:
meetings_dim=(meetings_dim.withColumnRenamed("meeting_id","openf1_meeting_id")
             .withColumnRenamed("race_id","kaggle_race_id")
             .withColumnRenamed("meeting_key","race_meeting_id"))

In [0]:
meetings_dim.show(5)

In [0]:
#check primary key uniqueness
meetings_dim.groupBy("race_meeting_id").count().filter(col("count") > 1).show()

# check natural key uniqueness
meetings_dim.groupBy("circuit_id", "year", "round").count().filter(col("count") > 1).show()

# check nulls
print(meetings_dim.filter(col("race_meeting_id").isNull()).count())
print(meetings_dim.filter(col("meeting_name").isNull()).count())
print(meetings_dim.filter(col("circuit_id").isNull()).count())

In [0]:
#write to gold schema

meetings_dim.write.format("delta").mode("overwrite").saveAsTable("f1_warehouse.gold.dim_meetings")